# PyTorch tensors: a course reference

A tensor is a multidimensional array whose **shape**, **dtype**, **device**, and
relationship to an automatic-differentiation graph determine how it behaves.
This notebook explains the tensor concepts and operations used by the PyTorch
MNIST and physics-informed neural-network (PINN) examples in this repository.

It is a reference rather than an exhaustive API catalogue. The examples are
small, require no downloaded data, and are intended to make the mechanisms
behind the course notebooks visible.

## Learning objectives

After working through this notebook, you should be able to:

- interpret tensor dimensions in terms of batches, channels, features, and
  physical coordinates;
- predict the result shape of indexing, broadcasting, reduction, reshaping,
  stacking, and concatenation operations;
- distinguish views, copies, detached tensors, and NumPy arrays that share
  storage;
- preserve dtype and device when creating or converting tensors;
- explain how `requires_grad`, `backward`, `torch.autograd.grad`, and gradient
  modes are used in the training and PINN notebooks.

**Required prerequisite:** basic Python and NumPy array operations.

**Helpful, but not required:** elementary derivatives and the chain rule.

## 1. Setup

In [1]:
import numpy as np
import torch

torch.manual_seed(1729)

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.13.0


## 2. The tensor contract

When reading tensor code, first establish five facts:

1. What does each dimension mean?
2. What is the element dtype?
3. On which device is the data stored?
4. Is PyTorch recording operations for automatic differentiation?
5. Does this tensor share storage with another object?

The first four can be inspected directly. Storage sharing is discussed in the
sections on views, copies, and NumPy conversion.

In [2]:
def describe(tensor):
    return {
        "shape": tuple(tensor.shape),
        "ndim": tensor.ndim,
        "dtype": tensor.dtype,
        "device": tensor.device,
        "requires_grad": tensor.requires_grad,
        "is_leaf": tensor.is_leaf,
        "grad_fn": (
            type(tensor.grad_fn).__name__
            if tensor.grad_fn is not None
            else None
        ),
    }

In [3]:
images = torch.rand(32, 1, 28, 28)
logits = torch.randn(32, 10)
pinn_inputs = torch.rand(512, 5)

describe(images), describe(logits), describe(pinn_inputs)

({'shape': (32, 1, 28, 28),
  'ndim': 4,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (32, 10),
  'ndim': 2,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (512, 5),
  'ndim': 2,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None})

The examples use these dimension conventions:

- `images`: `[batch, channel, height, width]`;
- `logits`: `[batch, class]`;
- `pinn_inputs`: `[sample, coordinate_or_parameter]`.

`shape` reports all dimension sizes, `ndim` reports the number of dimensions,
`size(dim)` reports one dimension size, and `numel()` reports the total number
of elements.

In [4]:
images.shape, images.ndim, images.size(0), images.numel()

(torch.Size([32, 1, 28, 28]), 4, 32, 25088)

## 3. Creating tensors, dtypes, and devices

`torch.tensor` copies values from Python or array-like input. Factory functions
such as `zeros`, `rand`, `arange`, and `linspace` construct tensors directly.
Specify `dtype` and `device` when they matter rather than relying on implicit
defaults.

In [5]:
from_python = torch.tensor(
    [[1.0, 2.0], [3.0, 4.0]],
    dtype=torch.float64,
)
zeros = torch.zeros(2, 3)
random_normal = torch.randn(2, 3)
indices = torch.arange(0, 10, 2)
coordinates = torch.linspace(0.0, 1.0, 6)

describe(from_python), indices, coordinates

({'shape': (2, 2),
  'ndim': 2,
  'dtype': torch.float64,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 tensor([0, 2, 4, 6, 8]),
 tensor([0.0000, 0.2000, 0.4000, 0.6000, 0.8000, 1.0000]))

### Dtype inference is source-dependent

Python floating-point values normally use PyTorch's current default floating
dtype, usually `torch.float32`. Python integers normally produce an integer
tensor. `torch.from_numpy` preserves the NumPy dtype, which is often
`float64`. Class labels in the MNIST notebooks are integers, while model inputs
and parameters are floating point.

In [6]:
python_floats = torch.tensor([1.0, 2.0])
python_integers = torch.tensor([1, 2])
numpy_values = np.array([1.0, 2.0])
from_numpy = torch.from_numpy(numpy_values)

python_floats.dtype, python_integers.dtype, from_numpy.dtype

(torch.float32, torch.int64, torch.float64)

`torch.empty` allocates storage without initializing its values. It is useful
when every element will be overwritten immediately, but it must not be treated
as a tensor of zeros.

In [7]:
samples = torch.empty(5)
samples.uniform_(-1.0, 1.0)

tensor([-0.4450, -0.1404,  0.7986, -0.5439, -0.5952])

### Constructing relative to another tensor

The `_like` factories and `new_tensor` are useful in device-independent code.
They inherit dtype and device from a reference tensor. `new_tensor` always
copies its input; when its input is already a tensor, explicit
`detach().clone()` is usually clearer.

In [8]:
reference = torch.linspace(0.0, 1.0, 4, dtype=torch.float64)
zeros_like_reference = torch.zeros_like(reference)
boundary_values = torch.full_like(reference, 2.5)
initial_time = reference.new_tensor([0.0])

(
    describe(reference),
    describe(zeros_like_reference),
    describe(boundary_values),
    describe(initial_time),
)

({'shape': (4,),
  'ndim': 1,
  'dtype': torch.float64,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (4,),
  'ndim': 1,
  'dtype': torch.float64,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (4,),
  'ndim': 1,
  'dtype': torch.float64,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (1,),
  'ndim': 1,
  'dtype': torch.float64,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None})

The older `torch.Tensor(...)` and `torch.FloatTensor(...)` constructors occur
in some PINN examples. Prefer `torch.tensor(...)` for existing data and factory
functions with explicit `dtype` and `device` for new data:

```python
samples = torch.empty(batch_size, device=device).uniform_(*domain)
```

## 4. Arithmetic, matrix multiplication, and broadcasting

Arithmetic operators are element-wise. Matrix multiplication uses `@` or
`torch.matmul`; it is not the same as element-wise multiplication.

In [9]:
left = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
right = torch.tensor([[0.5, 1.5], [2.0, -1.0]])

elementwise_product = left * right
matrix_product = left @ right

elementwise_product, matrix_product

(tensor([[ 0.5000,  3.0000],
         [ 6.0000, -4.0000]]),
 tensor([[ 4.5000, -0.5000],
         [ 9.5000,  0.5000]]))

### Broadcasting

Broadcasting compares dimensions from right to left. Two dimensions are
compatible when they are equal, one of them is `1`, or one dimension is absent.
Broadcasting logically expands compatible dimensions without copying all
values.

The PINN input normalization is a useful example: a `[sample, feature]` tensor
is combined with one lower and upper bound per feature.

In [10]:
inputs = torch.rand(8, 5)
lower = torch.tensor([0.0, 0.0, -1.0, -1.0, 0.05])
upper = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.20])

scaled_inputs = 2.0 * (inputs - lower) / (upper - lower) - 1.0

inputs.shape, lower.shape, scaled_inputs.shape

(torch.Size([8, 5]), torch.Size([5]), torch.Size([8, 5]))

**Reasoning check:** before running the next cell, predict the result shape.
This operation is legal, but is it likely to express an intended scientific
calculation?

In [11]:
column = torch.ones(4, 1)
row = torch.arange(4.0)
(column + row).shape

torch.Size([4, 4])

The result is `[4, 4]`, not `[4, 1]` or `[4]`. Broadcasting errors are often
semantic rather than runtime errors: an operation can succeed while computing
the wrong quantity.

Methods ending in an underscore, such as `add_` and `uniform_`, modify a tensor
in place. In-place operations can be efficient, but they also affect every view
sharing the same storage and can invalidate values saved for autograd. Prefer
ordinary out-of-place arithmetic unless mutation is deliberate.

In [12]:
original = torch.tensor([1.0, 2.0, 3.0])
modified = original.clone()
modified.subtract_(1.0)

original, modified

(tensor([1., 2., 3.]), tensor([0., 1., 2.]))

## 5. Indexing and masking

In [13]:
matrix = torch.arange(12).reshape(3, 4)

scalar = matrix[1, 2]
row = matrix[1]
column = matrix[:, 1]
dimension_preserving_column = matrix[:, 1:2]

(
    scalar.shape,
    row.shape,
    column.shape,
    dimension_preserving_column.shape,
)

(torch.Size([]), torch.Size([4]), torch.Size([3]), torch.Size([3, 1]))

Comparisons produce Boolean tensors. A Boolean mask can select or replace
elements without writing a Python loop.

In [14]:
values = torch.tensor([-2.0, -0.5, 0.0, 1.5, 3.0])
negative_mask = values < 0.0
positive_values = values.clone()
positive_values[negative_mask] = 0.0

negative_mask, values[negative_mask], positive_values

(tensor([ True,  True, False, False, False]),
 tensor([-2.0000, -0.5000]),
 tensor([0.0000, 0.0000, 0.0000, 1.5000, 3.0000]))

Basic indexing usually returns a view that shares storage. Advanced indexing,
including indexing with a tensor of positions, returns a copy. This difference
matters when values are modified later.

In [15]:
source = torch.arange(6.0)
basic_view = source[1:4]
advanced_copy = source[torch.tensor([1, 2, 3])]

source[2] = -1.0

source, basic_view, advanced_copy

(tensor([ 0.,  1., -1.,  3.,  4.,  5.]),
 tensor([ 1., -1.,  3.]),
 tensor([1., 2., 3.]))

## 6. Reshaping, views, and contiguity

A view has different shape or indexing metadata but shares the underlying
storage. `view` can only reinterpret compatible strides. `reshape` returns a
view when possible and otherwise makes a copy; code should not rely on which
choice it makes.

In [16]:
flat = torch.arange(12.0)
grid = flat.view(3, 4)
grid[1, 1] = -1.0

flat, grid

(tensor([ 0.,  1.,  2.,  3.,  4., -1.,  6.,  7.,  8.,  9., 10., 11.]),
 tensor([[ 0.,  1.,  2.,  3.],
         [ 4., -1.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [17]:
transposed = grid.transpose(0, 1)

try:
    flattened_with_view = transposed.view(-1)
except RuntimeError as error:
    flattened_with_view = str(error).splitlines()[0]

flattened_with_reshape = transposed.reshape(-1)

(
    transposed.is_contiguous(),
    flattened_with_view,
    flattened_with_reshape,
)

(False,
 "view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.",
 tensor([ 0.,  4.,  8.,  1., -1.,  9.,  2.,  6., 10.,  3.,  7., 11.]))

`unsqueeze` inserts a dimension of size one. `squeeze(dim)` removes a specific
size-one dimension. Supplying `dim` is safer than an unrestricted `squeeze()`:
it documents which dimension is expected to disappear and avoids accidentally
removing a batch dimension of size one.

In [18]:
image = torch.rand(28, 28)
image_with_channel = image.unsqueeze(0)
batch = image_with_channel.unsqueeze(0)
image_again = batch.squeeze(0).squeeze(0)

image.shape, image_with_channel.shape, batch.shape, image_again.shape

(torch.Size([28, 28]),
 torch.Size([1, 28, 28]),
 torch.Size([1, 1, 28, 28]),
 torch.Size([28, 28]))

### Combining and splitting tensors

`cat` joins tensors along an existing dimension. `stack` introduces a new
dimension. `column_stack` constructs columns, while `unbind` removes one
dimension and returns its slices as separate tensors.

In [19]:
first = torch.tensor([1.0, 2.0])
second = torch.tensor([3.0, 4.0])

concatenated = torch.cat([first, second])
stacked = torch.stack([first, second])
columns = torch.column_stack([first, second])

concatenated.shape, stacked.shape, columns.shape

(torch.Size([4]), torch.Size([2, 2]), torch.Size([2, 2]))

In [20]:
coordinates = torch.rand(6, 5)
x, t, coefficient_1, coefficient_2, diffusivity = coordinates.unbind(dim=1)

x.shape, t.shape, diffusivity.shape

(torch.Size([6]), torch.Size([6]), torch.Size([6]))

### Constructing coordinate grids

The heat-equation PINN evaluates combinations of time and position.
`meshgrid(..., indexing="ij")` constructs coordinate matrices, `ravel()`
flattens them, and `column_stack` turns the coordinates into one row per
evaluation point.

In [21]:
x_grid = torch.linspace(0.0, 1.0, 4)
t_grid = torch.linspace(0.0, 0.5, 3)
tt, xx = torch.meshgrid(t_grid, x_grid, indexing="ij")
grid_points = torch.column_stack([xx.ravel(), tt.ravel()])

xx.shape, tt.shape, grid_points.shape, grid_points[:4]

(torch.Size([3, 4]),
 torch.Size([3, 4]),
 torch.Size([12, 2]),
 tensor([[0.0000, 0.0000],
         [0.3333, 0.0000],
         [0.6667, 0.0000],
         [1.0000, 0.0000]]))

## 7. Reductions and scalar tensors

Reductions combine values along one or more dimensions. The `dim` argument
specifies which dimension disappears; `keepdim=True` retains it with size one,
which can make subsequent broadcasting clearer.

In [22]:
measurements = torch.tensor(
    [[3.0, 1.0, 4.0], [2.0, 5.0, 0.0]]
)

global_minimum = measurements.min()
column_minima = measurements.min(dim=0)
row_means = measurements.mean(dim=1)
row_means_kept = measurements.mean(dim=1, keepdim=True)

(
    global_minimum,
    column_minima.values,
    column_minima.indices,
    row_means.shape,
    row_means_kept.shape,
)

(tensor(0.),
 tensor([2., 1., 0.]),
 tensor([1, 0, 1]),
 torch.Size([2]),
 torch.Size([2, 1]))

The MNIST training notebooks turn class scores into predictions and Boolean
comparisons into a count of correct examples.

In [23]:
targets = torch.tensor([2, 0, 1])
example_logits = torch.tensor(
    [
        [0.1, 0.2, 1.7],
        [1.2, 0.4, 0.3],
        [0.1, 1.3, 0.2],
    ]
)

predictions = example_logits.argmax(dim=1)
correct = predictions == targets
number_correct = correct.sum()
accuracy = correct.float().mean()

predictions, correct, number_correct, accuracy

(tensor([2, 0, 1]), tensor([True, True, True]), tensor(3), tensor(1.))

`torch.bincount` counts occurrences of non-negative integer values.
`minlength` retains classes that are absent from the observed data, which is
why the MNIST notebooks specify the known number of classes.

In [24]:
labels = torch.tensor([0, 1, 1, 3, 3, 3])
class_counts = torch.bincount(labels, minlength=5)
class_counts

tensor([1, 2, 0, 3, 0])

Scientific losses and error measures often compose element-wise operations and
reductions. `torch.linalg.vector_norm` makes the intended norm explicit.

In [25]:
residual = torch.tensor([-0.2, 0.1, 0.3, -0.1])

mean_squared_residual = residual.square().mean()
root_mean_squared_residual = mean_squared_residual.sqrt()
l2_norm = torch.linalg.vector_norm(residual)
maximum_absolute_residual = residual.abs().max()

(
    mean_squared_residual,
    root_mean_squared_residual,
    l2_norm,
    maximum_absolute_residual,
)

(tensor(0.0375), tensor(0.1936), tensor(0.3873), tensor(0.3000))

A reduction still returns a tensor, even when it has no dimensions. `item()`
converts a one-element tensor to a Python number. It is not differentiable and,
on an accelerator, obtaining the value can force the host to wait for queued
device work. Use it for reporting or control flow, not repeatedly inside
performance-critical tensor calculations.

In [26]:
scalar_tensor = residual.square().mean()
python_number = scalar_tensor.item()

scalar_tensor.shape, type(scalar_tensor), type(python_number)

(torch.Size([]), torch.Tensor, float)

## 8. Copies, detaching, and NumPy interoperability

`clone()` copies tensor data while preserving its relationship to the autograd
graph. `detach()` returns a tensor that shares storage but is disconnected from
the graph. Use `detach().clone()` when both an independent copy and graph
disconnection are required.

`describe()` reports metadata and autograd status; it cannot reveal whether two
tensors share storage. The mutation below makes that difference observable.

In [27]:
tracked = torch.tensor([1.0, 2.0], requires_grad=True)
result = tracked.square()
cloned_result = result.clone()
detached = tracked.detach()
independent = tracked.detach().clone()

descriptions = {
    "result": describe(result),
    "cloned result": describe(cloned_result),
    "detached": describe(detached),
    "detached clone": describe(independent),
}

with torch.no_grad():
    tracked[0] = -1.0

(
    descriptions,
    tracked,
    detached,      # changed because it shares storage with tracked
    independent,   # unchanged because clone() copied the data
)

({'result': {'shape': (2,),
   'ndim': 1,
   'dtype': torch.float32,
   'device': device(type='cpu'),
   'requires_grad': True,
   'is_leaf': False,
   'grad_fn': 'PowBackward0'},
  'cloned result': {'shape': (2,),
   'ndim': 1,
   'dtype': torch.float32,
   'device': device(type='cpu'),
   'requires_grad': True,
   'is_leaf': False,
   'grad_fn': 'CloneBackward0'},
  'detached': {'shape': (2,),
   'ndim': 1,
   'dtype': torch.float32,
   'device': device(type='cpu'),
   'requires_grad': False,
   'is_leaf': True,
   'grad_fn': None},
  'detached clone': {'shape': (2,),
   'ndim': 1,
   'dtype': torch.float32,
   'device': device(type='cpu'),
   'requires_grad': False,
   'is_leaf': True,
   'grad_fn': None}},
 tensor([-1.,  2.], requires_grad=True),
 tensor([-1.,  2.]),
 tensor([1., 2.]))

`torch.from_numpy` and a supported CPU tensor's `.numpy()` method normally
share storage. `torch.tensor(array)` copies instead. Mutating shared data from
either side changes the other object.

In [28]:
array = np.arange(4, dtype=np.float32)
shared_tensor = torch.from_numpy(array)
copied_tensor = torch.tensor(array)

array[0] = -1.0

array, shared_tensor, copied_tensor

(array([-1.,  1.,  2.,  3.], dtype=float32),
 tensor([-1.,  1.,  2.,  3.]),
 tensor([0., 1., 2., 3.]))

The robust conversion used for plotting or NumPy analysis is:

```python
array = tensor.detach().cpu().numpy()
```

- `detach()` removes the autograd relationship;
- `cpu()` transfers accelerator data to host memory;
- `numpy()` exposes the CPU values as an array.

The resulting array may still share storage with the detached CPU tensor. Add
`.copy()` on the NumPy side when independent storage is required.

In [29]:
plot_values = result.detach().cpu().numpy()
plot_values

array([1., 4.], dtype=float32)

## 9. Moving tensors between devices

Operations generally require participating tensors to be on the same device.
`to` can change device, dtype, or both. It returns the original tensor when no
conversion is needed and otherwise returns a converted tensor; assigning the
result is therefore essential.

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
using_cuda = device.type == "cuda"

host_values = torch.linspace(0.0, 1.0, 5)
device_values = host_values.to(
    device,
    dtype=torch.float32,
    non_blocking=using_cuda,
)
returned_values = device_values.cpu()

describe(host_values), describe(device_values), describe(returned_values)

({'shape': (5,),
  'ndim': 1,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (5,),
  'ndim': 1,
  'dtype': torch.float32,
  'device': device(type='cuda', index=0),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (5,),
  'ndim': 1,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': False,
  'is_leaf': True,
  'grad_fn': None})

The MNIST loaders use `pin_memory=True` only when CUDA is used, and tensor
transfers separately use `non_blocking=True`. Pinned host memory can enable
efficient asynchronous host-to-device copies, but it consumes a limited host
resource and does not guarantee a speedup. It is a data-loading optimization,
not a requirement for correctness.

Create tensors directly on their target device when possible:

In [31]:
device_samples = torch.rand(4, 5, device=device)
describe(device_samples)

{'shape': (4, 5),
 'ndim': 2,
 'dtype': torch.float32,
 'device': device(type='cuda', index=0),
 'requires_grad': False,
 'is_leaf': True,
 'grad_fn': None}

## 10. Automatic differentiation

PyTorch records operations involving tensors that require gradients. The
recorded graph is created dynamically during each forward calculation. It
enables reverse-mode automatic differentiation but consumes time and memory.

A **leaf tensor** is not the result of a recorded differentiable operation.
Model parameters and explicitly marked input coordinates are typical leaves.
Only leaf tensors have `.grad` populated by `backward()` unless
`retain_grad()` is requested for an intermediate tensor.

In [32]:
x = torch.tensor([2.0, 3.0, 4.0], requires_grad=True)
u = 3.0 * x**2 - 2.0 * x + 1.0
loss = u.sum()

describe(x), describe(u), describe(loss)

({'shape': (3,),
  'ndim': 1,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': True,
  'is_leaf': True,
  'grad_fn': None},
 {'shape': (3,),
  'ndim': 1,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': True,
  'is_leaf': False,
  'grad_fn': 'AddBackward0'},
 {'shape': (),
  'ndim': 0,
  'dtype': torch.float32,
  'device': device(type='cpu'),
  'requires_grad': True,
  'is_leaf': False,
  'grad_fn': 'SumBackward0'})

`backward()` traverses the graph from a scalar result and accumulates
derivatives in the `.grad` attributes of leaf tensors.

In [33]:
loss.backward()
first_gradient = x.grad.clone()

loss = (3.0 * x**2 - 2.0 * x + 1.0).sum()
loss.backward()
accumulated_gradient = x.grad.clone()

first_gradient, accumulated_gradient

(tensor([10., 16., 22.]), tensor([20., 32., 44.]))

The second result is twice the first because gradients accumulate. Neural
network training loops call `optimizer.zero_grad(...)` before the next backward
pass. For an isolated leaf tensor, assigning `x.grad = None` clears it.

In [34]:
x.grad = None
x.grad

### `backward()` versus `torch.autograd.grad`

`backward()` is convenient for accumulating a loss gradient in all model
parameters. `torch.autograd.grad` returns derivatives with respect to selected
inputs. PINNs use the latter to differentiate the network output with respect
to space and time.

In [35]:
coordinate = torch.linspace(-1.0, 1.0, 5).requires_grad_()
field = coordinate**3

first_derivative = torch.autograd.grad(
    field,
    coordinate,
    grad_outputs=torch.ones_like(field),
    create_graph=True,
)[0]
second_derivative = torch.autograd.grad(
    first_derivative,
    coordinate,
    grad_outputs=torch.ones_like(first_derivative),
)[0]

coordinate, first_derivative, second_derivative

(tensor([-1.0000, -0.5000,  0.0000,  0.5000,  1.0000], requires_grad=True),
 tensor([3.0000, 0.7500, 0.0000, 0.7500, 3.0000], grad_fn=<MulBackward0>),
 tensor([-6., -3.,  0.,  3.,  6.]))

`grad_outputs` defines a vector-Jacobian product. With
`torch.ones_like(field)`, it differentiates the sum of the output components.
The result above can be read as one derivative per sample because each output
depends only on the corresponding input. That interpretation is not generally
valid for operations that couple different samples.

`create_graph=True` records the derivative calculation itself. It is required
when another derivative will be taken, and when a PINN loss must be
differentiated through a physical derivative back to model parameters.

### Gradient modes and model evaluation

`torch.no_grad()` and `torch.inference_mode()` prevent operations from being
recorded by autograd. Inference mode removes additional tracking overhead but
places more restrictions on using its results later in differentiated code.

`model.eval()` is separate: it changes the behaviour of modules such as
dropout and batch normalization but does not disable gradient recording. The
MNIST evaluation functions correctly use both `model.eval()` and
`@torch.inference_mode()`.

In [36]:
tracked_input = torch.tensor([2.0], requires_grad=True)

ordinary_output = tracked_input.square()
with torch.no_grad():
    no_grad_output = tracked_input.square()
with torch.inference_mode():
    inference_output = tracked_input.square()

(
    ordinary_output.requires_grad,
    no_grad_output.requires_grad,
    inference_output.requires_grad,
)

(True, False, False)

## 11. Appendix: functional differentiation used by the PINNs

`torch.func` provides function transformations rather than tensor methods:

- `grad(f)` returns a function that evaluates the gradient of scalar-valued
  `f`;
- transformations can be nested for higher derivatives;
- `vmap(f)` maps `f` over a tensor dimension without writing a Python loop;
- `functional_call` evaluates a module with an explicit parameter mapping.

This style is used by the logistic and pendulum PINNs. It is useful for batched
derivatives, but is a more advanced and more version-sensitive API than the
core tensor and autograd mechanisms above.

In [37]:
from torch.func import grad, vmap


def scalar_function(value):
    return torch.sin(value)


first_derivative_function = grad(scalar_function)
second_derivative_function = grad(first_derivative_function)
points = torch.linspace(-1.0, 1.0, 5)

(
    vmap(first_derivative_function)(points),
    vmap(second_derivative_function)(points),
)

(tensor([0.5403, 0.8776, 1.0000, 0.8776, 0.5403]),
 tensor([ 0.8415,  0.4794, -0.0000, -0.4794, -0.8415]))

The PINN notebooks use `functional_call` to evaluate a module with
an explicit parameter mapping. This makes the parameters ordinary function
arguments that can participate in function transformations.

In [38]:
from torch import nn
from torch.func import functional_call

model = nn.Sequential(
    nn.Linear(1, 4),
    nn.Tanh(),
    nn.Linear(4, 1),
)
parameters = dict(model.named_parameters())


def model_at(value, model_parameters):
    output = functional_call(
        model,
        model_parameters,
        (value.reshape(1, 1),),
    )
    return output.squeeze()


model_derivative = grad(model_at)
model_points = torch.linspace(-1.0, 1.0, 5)
vmap(model_derivative, in_dims=(0, None))(model_points, parameters)

tensor([ 0.0353,  0.0208,  0.0016, -0.0171, -0.0318], grad_fn=<ViewBackward0>)

## 12. Reasoning checks

Try to answer these before opening the solutions.

1. An MNIST batch has shape `[64, 1, 28, 28]`. What are the shapes of
   `batch[0]`, `batch[0:1]`, and `batch.squeeze(1)`?
2. Why can `tensor.reshape(...)` be safer than `tensor.view(...)`, and why must
   code still avoid relying on whether `reshape` returns a view or a copy?
3. What is the difference between `torch.cat([a, b])` and
   `torch.stack([a, b])` for one-dimensional `a` and `b`?
4. Why is `prediction.detach().cpu().numpy()` preferable to
   `prediction.numpy()` in device-independent model code?
5. Why does a PINN need `create_graph=True` while calculating some physical
   derivatives?

<details>
<summary>Solutions</summary>

1. `[1, 28, 28]`, `[1, 1, 28, 28]`, and `[64, 28, 28]`. Integer indexing
   removes a dimension, slicing preserves it, and `squeeze(1)` removes the
   known size-one channel dimension.
2. `view` requires compatible strides, while `reshape` can copy when needed.
   Because `reshape` may either alias or copy, mutation-sensitive code should
   use an explicit view or `clone()` according to its intended ownership.
3. `cat` extends the existing dimension; `stack` introduces a new dimension.
4. It works for tensors that track gradients and for accelerator tensors. The
   direct `.numpy()` conversion is restricted to supported CPU tensors that do
   not require gradients.
5. The derivative must itself remain in the autograd graph so that a higher
   derivative can be computed or the physics loss can propagate back to model
   parameters.

</details>

## 13. Further reference

- [Tensor attributes](https://docs.pytorch.org/docs/stable/tensor_attributes.html)
- [Broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
- [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [`torch.func`](https://docs.pytorch.org/docs/stable/func.html)

The official documentation is the authority for version-specific signatures.
This notebook focuses on the durable concepts needed to understand the course
examples.